# Advance Crime Data Pipeline

Stage 1: Ingestion Layer

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 1 of 5: Ingestion <br>
**Medallion Layer:** Bronze 🥉 <br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Dyfed-Powys <br>
**Authors:** Group 1 <br>
**Last Updated:** 18 May 2026 


## 1. Environment Setup

In [ ]:
-- Create a warehouse for compute
CREATE WAREHOUSE IF NOT EXISTS CRIME_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

-- Create the database
CREATE DATABASE IF NOT EXISTS CRIME_PIPELINE;

-- Create schemas for each pipeline stage
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.RAW;
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.CLEAN;
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.REPORTING;

CREATE STAGE IF NOT EXISTS CRIME_PIPELINE.RAW.CRIME_STAGE;


In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

STAGE        = "@CRIME_PIPELINE.RAW.CRIME_STAGE"
BRONZE_TABLE = "CRIME_PIPELINE.RAW.BRONZE_CRIME_RAW"

SELECTED_FORCES = [
    "west-midlands",
    "thames-valley",
    "surrey",
    "dyfed-powys"
]

print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. File Discovery

In [ ]:
list_df = session.sql(f"LIST {STAGE}").to_pandas()
list_df.columns = [c.strip().strip('"').lower() for c in list_df.columns]

# Retain only files belonging to the four selected forces
list_df["stem"] = list_df["name"].str.split("/").str[-1]
list_df = list_df[
    list_df["stem"].apply(lambda f: any(force in f for force in SELECTED_FORCES))
].reset_index(drop=True)

# Derive month from filename prefix e.g. '2026-01-surrey-street.csv' -> '2026-01'
list_df["month"] = list_df["stem"].str[:7]

months = sorted(list_df["month"].unique())

print(f"{len(list_df)} file(s) identified across {len(months)} month(s):")
print(list_df[["month", "stem", "size"]].to_string(index=False))

## 3. Test Batch

In [ ]:
TEST_MONTHS = [months[0]]  # first available month only

batch_frames = []

for month in TEST_MONTHS:
    month_files = list_df[list_df["month"] == month]["stem"].tolist()

    for filename in month_files:
        df = pd.read_csv(
            session.file.get_stream(f"{STAGE}/{filename}"),
            dtype=str,
            low_memory=False
        )

        df["source_month"] = month
        df["source_file"]  = filename

        batch_frames.append(df)
        print(f"Loaded: {filename} — {len(df):,} rows")

crime_raw = pd.concat(batch_frames, ignore_index=True)

print(f"\nTest batch complete: {len(crime_raw):,} rows, {crime_raw['source_file'].nunique()} file(s)")

In [ ]:
# Quick validation checks after ingestion layer complete

print("Total rows:", len(crime_raw))
print("Files loaded:", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

## 4. Test Batch Inspection

In [ ]:
# Inspecting the data
print("Total rows   :", len(crime_raw))
print("Files loaded :", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

crime_raw.head()

## 5. Full Ingestion

In [ ]:
batch_frames = []

for month in months:
    month_files = list_df[list_df["month"] == month]["stem"].tolist()

    for filename in month_files:
        df = pd.read_csv(
            session.file.get_stream(f"{STAGE}/{filename}"),
            dtype=str,
            low_memory=False
        )

        df["source_month"] = month
        df["source_file"]  = filename

        batch_frames.append(df)
        print(f"Loaded: {filename} — {len(df):,} rows")

crime_raw = pd.concat(batch_frames, ignore_index=True)

print(f"\nFull ingestion complete: {len(crime_raw):,} rows across {crime_raw['source_month'].nunique()} month(s)")

In [ ]:
print("Total rows   :", len(crime_raw))
print("Files loaded :", crime_raw["source_file"].nunique())
print("Months loaded:", crime_raw["source_month"].nunique())

print("\nRows per month:")
print(crime_raw["source_month"].value_counts().sort_index())

print("\nRows per force file:")
print(crime_raw["source_file"].value_counts())

6. Export Ingested Data to Bronze Table

In [ ]:
from snowflake.snowpark.functions import col

# Write directly using write_pandas — bypasses temp stage creation
from snowflake.connector.pandas_tools import write_pandas

# Get the underlying connector connection from the Snowpark session
conn = session.connection

success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=crime_raw,
    table_name="BRONZE_CRIME_RAW",
    database="CRIME_PIPELINE",
    schema="RAW",
    auto_create_table=True,
    overwrite=True
)

print(f"Bronze table written successfully")
print(f"Table  : CRIME_PIPELINE.RAW.BRONZE_CRIME_RAW")
print(f"Rows   : {nrows:,}")
print(f"Chunks : {nchunks}")